# 6 · Graph schema — declare the shape, then have Postgres hold you to it

Notebook 05 constrained *properties*. This one declares the shape of the whole
graph — which node types exist, what each carries, and which edge kinds connect
which node types — and then, as a separate and explicit step, compiles that
declaration into constraints the database enforces on every write path.

Three things are deliberately separate here, and the separation is the design:

1. **Declaring** (`define_schema`) is in-memory and instant. Nothing is validated.
2. **Enforcing** (`enforce_schema`) is DDL. `ADD CONSTRAINT` validates every
   existing row, which can fail on data that grew before the schema did.
3. **Inferring** (`infer_schema`) is an *observation* of the rows you already have.
   Adopting it is your call, in a separate line.

In [1]:
from dataclasses import dataclass
from typing import Optional

from demo_graph import connect, seed
from hopai import ConstraintViolation, Start

graph = connect("nb_06_schema")
seed(graph)
print(graph.schema, "<- nothing declared yet")

None <- nothing declared yet


## Declaring, in classes

Plain dataclasses. A field with no default is required; `Optional[X]` is optional
*and* may be JSON null. An **edge** class names its endpoints as fields annotated
with node classes — the way an ORM association object does — and everything else on
it is a property.

In [2]:
@dataclass
class Person:
    email: str                  # no default -> required
    name: str
    age: Optional[int] = None   # optional, may be null
    city: Optional[str] = None
    active: Optional[bool] = None


@dataclass
class Company:
    name: str
    founded: Optional[int] = None


@dataclass
class Friend:
    source: Person              # endpoint, not a property
    target: Person


@dataclass
class WorksAt:
    source: Person
    target: Company
    since: int                  # property


graph.define_schema(nodes=[Person, Company], edges=[Friend, WorksAt])

GraphSchema(node_types=(NodeType(name='person', properties=(Property(name='email', json_type=('string',), required=True, unique=False, values=(), format=None, properties=()), Property(name='name', json_type=('string',), required=True, unique=False, values=(), format=None, properties=()), Property(name='age', json_type=('null', 'number'), required=False, unique=False, values=(), format=None, properties=()), Property(name='city', json_type=('null', 'string'), required=False, unique=False, values=(), format=None, properties=()), Property(name='active', json_type=('boolean', 'null'), required=False, unique=False, values=(), format=None, properties=()))), NodeType(name='company', properties=(Property(name='name', json_type=('string',), required=True, unique=False, values=(), format=None, properties=()), Property(name='founded', json_type=('null', 'number'), required=False, unique=False, values=(), format=None, properties=())))), edge_types=(EdgeType(kind='friend', source='person', target='p

Class names become snake_case type names — `WorksAt` → `works_at` — matching how
the Cypher examples spell kinds. The annotation mapping is one deterministic table:

```
str        -> "string"      dict / dict[...]  -> "object"
int, float -> "number"      list / list[...]  -> "array"
bool       -> "boolean"     Optional[X]       -> X's type + "null", not required
```

An `Enum`, a `datetime`/`date` and a nested dataclass map too — with the extra
detail they carry, covered further down. What is left over, a union that is not
`Optional` above all, is refused with the explicit-`Property` rewrite named rather
than coerced into a guess.

Pydantic v2 models work anywhere a dataclass does. And the same schema can be
spelled with primitives when there is no class to hand over — all three inputs
normalize to the identical canonical form:

```python
graph.define_schema(
    nodes=[NodeType("person", properties=[Property("email", "string", required=True)])],
    edges=[EdgeType("works_at", source="person", target="company")],
)
```

## Reading it back, in whichever vocabulary the consumer speaks

In [3]:
import json

print(json.dumps(graph.schema_json, indent=2)[:700], "...")

{
  "nodes": {
    "person": {
      "type": "object",
      "properties": {
        "email": {
          "type": "string"
        },
        "name": {
          "type": "string"
        },
        "age": {
          "type": [
            "null",
            "number"
          ]
        },
        "city": {
          "type": [
            "null",
            "string"
          ]
        },
        "active": {
          "type": [
            "boolean",
            "null"
          ]
        }
      },
      "required": [
        "email",
        "name"
      ]
    },
    "company": {
      "type": "object",
      "properties": {
        "name": {
          "type": "string"
        },
         ...


`graph.schema_json` is JSON Schema vocabulary, `json.dumps`-clean — made to be
pasted into a system prompt or returned as a tool result, so a model writing
traversals knows what the properties are called before it guesses.

There are three more views of the same object: `graph.schema` (the canonical
dataclasses, and `None` until `define_schema` is called — that is the existence
check), `graph.schema_networkx` (a meta-graph of types), and
`graph.schema_pydantic` (generated models, one per type).

In [4]:
meta = graph.schema_networkx           # pip install hopai[networkx]
print(meta, "->", list(meta.edges(keys=True)))

models = graph.schema_pydantic         # pip install hopai[pydantic]
print(models)
print(models["person"](email="a@example.com", name="Ada"))

MultiDiGraph with 2 nodes and 2 edges -> [('person', 'person', 'friend'), ('person', 'company', 'works_at')]
{'person': <class 'hopai.schema.Person'>, 'company': <class 'hopai.schema.Company'>, 'friend': <class 'hopai.schema.Friend'>, 'works_at': <class 'hopai.schema.WorksAt'>}
email='a@example.com' name='Ada' age=None city=None active=None


There is a fifth representation for the one consumer the others do not serve: a
human skimming a pull request. `schema_mermaid` needs no dependency at all —
GitHub, GitLab and most doc tooling render Mermaid natively, so the string goes
into a ```` ```mermaid ```` fence and becomes a picture.

In [5]:
print(graph.schema_mermaid)

flowchart LR
    person["person (email*, name*, age, city, active)"]
    company["company (name*, founded)"]
    person -- friend --> person
    person -- works_at --> company


Which renders as:

```mermaid
flowchart LR
    person["person (email*, name*, age, city, active)"]
    company["company (name*, founded)"]
    person -- friend --> person
    person -- works_at --> company
```

`*` marks required and `!` marks unique — the same markers `tool_summary()` puts in
a tool description, capped the same way on wide types. One arrow per
`(kind, source, target)` triple, so a kind connecting two different endpoint pairs
stays two arrows rather than collapsing into one.

## Enforcing

`schema_ddl()` shows the SQL first, as always. Presence and JSON type per node type
and edge kind compile to guarded CHECK constraints:

In [6]:
for statement in graph.schema_ddl()[:3]:
    print(statement, "\n")
print(f"... {len(graph.schema_ddl())} statements in total")

ALTER TABLE "nodes" ADD CONSTRAINT "ck_schema_req_default_person" CHECK (graph_id != 'default' OR (properties ->> 'type') IS DISTINCT FROM 'person' OR (properties ?& ARRAY['email', 'name'])) 

ALTER TABLE "nodes" ADD CONSTRAINT "ck_schema_typ_default_person_email" CHECK (graph_id != 'default' OR (properties ->> 'type') IS DISTINCT FROM 'person' OR jsonb_typeof(properties['email']) = 'string') 

ALTER TABLE "nodes" ADD CONSTRAINT "ck_schema_typ_default_person_name" CHECK (graph_id != 'default' OR (properties ->> 'type') IS DISTINCT FROM 'person' OR jsonb_typeof(properties['name']) = 'string') 

... 11 statements in total


The `IS DISTINCT FROM 'person'` guard is what makes each rule apply only to rows of
its own type — and vacuously true for rows carrying no `type` at all, so untyped
rows pass by construction rather than by accident.

Before running any of it on a graph that grew first, ask what would break.
`schema_violations()` is read-only and returns the whole work list, where
`ADD CONSTRAINT` would fail opaquely on the first bad row:

In [7]:
print(graph.schema_violations() or "no violations -- enforce_schema() would succeed")

no violations -- enforce_schema() would succeed


In [8]:
# Add a row that breaks the contract, then ask again.
graph.add_nodes([{"type": "person", "name": "Mallory", "age": "not a number"}])
report = graph.schema_violations()
print(bool(report))
print(report)

True
2 row(s) violate 2 schema rule(s):
  ck_schema_req_default_person (nodes): 1 row(s), e.g. id 8
  ck_schema_typ_default_person_age (nodes): 1 row(s), e.g. id 8


Two rules broken by one row: `email` is required and missing, and `age` is declared
`number` but holds a string. The report names the constraint that would be created,
how many rows fail it, and sample ids — the work list, not a first casualty.

Enforcing now fails, and names the rule the data breaks. Note the exception type:
a rejected **write** is translated into `ConstraintViolation`, but this is DDL
failing, so the driver's `IntegrityError` comes through as-is — one more reason to
ask `schema_violations()` first rather than learn about the data from a failed
migration:

In [9]:
try:
    graph.enforce_schema()
except Exception as exc:
    print(f"{type(exc).__name__}: {str(exc).splitlines()[0][:160]}")

IntegrityError: (psycopg2.errors.CheckViolation) check constraint "ck_schema_req_default_person" of relation "nodes" is violated by some row


There is no update API here yet, so the fix in this notebook is to drop the bad row
directly through SQLAlchemy — in a real system it is whatever your data-repair path
is, driven by the ids the report handed you.

In [10]:
from sqlalchemy import text

with graph.engine.begin() as conn:
    conn.execute(text("DELETE FROM nodes WHERE properties->>'name' = 'Mallory'"))

print(graph.schema_violations() or "clean")
applied = graph.enforce_schema()
print(f"{len(applied)} constraints in force, e.g. {applied[:2]}")

clean


11 constraints in force, e.g. ['ck_schema_req_default_person', 'ck_schema_typ_default_person_email']


Now every write path is validated by the server — `add_nodes`, `merge_nodes`, a
Cypher `CREATE`, or SQL from another service — and a violation surfaces as
`ConstraintViolation`:

In [11]:
for row in ({"type": "person", "name": "Nobody"},                    # no email
            {"type": "person", "name": "Ninety", "email": "n@example.com", "age": "90"}):
    try:
        graph.add_nodes([row])
        print("accepted (!)", row)
    except ConstraintViolation as exc:
        print(f"{exc.constraint}\n   {exc}")

ck_schema_req_default_person
   node rejected by constraint 'ck_schema_req_default_person' -- Failing row contains (9, default, {"name": "Nobody", "type": "person"}).
ck_schema_typ_default_person_age
   node rejected by constraint 'ck_schema_typ_default_person_age' -- Failing row contains (10, default, {"age": "90", "name": "Ninety", "type": "person", "email": "n@ex...).


`enforce_schema()` is idempotent **and reconciles**: re-running it after a schema
change drops the schema-derived constraints the current schema no longer produces,
so it converges instead of accreting. It only ever touches objects it named
(`ck_schema_*`) — your `define_constraints()` declarations are not its to drop.

## Endpoint types need a trigger, and cost something

"`works_at` connects a person to a company" cannot be a CHECK: a CHECK sees one
row, and this rule needs the endpoint *nodes*. So it is an opt-in backed by a
constraint trigger, priced per edge write — stated here rather than switched on
quietly.

In [12]:
graph.enforce_schema(endpoints=True)

# A `robot` node is fine on its own: every schema CHECK is guarded by type, so
# a type the schema never mentions passes by construction.
graph.add_nodes([{"type": "robot", "name": "R2"}])
robot = graph.traverse(Start(where={"name": "R2"})).nodes[0]["id"]
acme = graph.traverse(Start(where={"name": "Acme"})).nodes[0]["id"]

try:
    graph.add_edges([{"start_id": robot, "end_id": acme, "kind": "works_at", "since": 2024}])
except ConstraintViolation as exc:
    print(exc)

edge rejected by constraint 'ck_schema_end_default' -- works_at connects robot -> company, but the schema declares: friend: person -> person; works_at: person -> company


The message names what was written and what the schema declares. Note the limit
this has, and it is inherent rather than an oversight: the trigger validates edges
**as they are written**. Retyping a node under existing edges is not re-checked.

Calling `enforce_schema()` again without `endpoints=True` drops the trigger — the
same reconciliation as the CHECKs.

## More than presence and type

A `Property` also carries an allowed value set, a format, a nested object schema,
and per-type uniqueness. Most of that comes out of annotations you would write
anyway:

| Annotation | Property |
| --- | --- |
| an `Enum` with one value type | that type, plus `values=(...)` |
| `datetime` / `date` | `"string"` plus `format="date-time"` / `"date"` |
| a nested dataclass or model | `"object"` plus that class's own property schema |

In [13]:
from datetime import datetime
from enum import Enum


class Status(Enum):
    ACTIVE = "active"
    GONE = "gone"


@dataclass
class Address:
    city: str
    country: str


@dataclass
class Employee:
    email: str
    status: Status        # -> "string" + the allowed values
    joined: datetime      # -> "string" + a date-time format
    address: Address      # -> "object" + a nested property schema


richer = graph.in_graph("richer")
richer.define_schema(nodes=[Employee])
print(json.dumps(richer.schema_json["nodes"]["employee"], indent=2))

{
  "type": "object",
  "properties": {
    "email": {
      "type": "string"
    },
    "status": {
      "type": "string",
      "enum": [
        "active",
        "gone"
      ]
    },
    "joined": {
      "type": "string",
      "format": "date-time"
    },
    "address": {
      "type": "object",
      "properties": {
        "city": {
          "type": "string"
        },
        "country": {
          "type": "string"
        }
      },
      "required": [
        "city",
        "country"
      ]
    }
  },
  "required": [
    "email",
    "status",
    "joined",
    "address"
  ]
}


Note where each of those is *enforced*, because the two halves differ on purpose.
The database checks the JSON type of the stored value — `jsonb_typeof` — and, for a
value set, that the text matches one of the allowed values. It does not parse ISO
timestamps or descend into a nested object, because a CHECK that did would be a
schema validator written in SQL.

The generated pydantic models do that half, which is the point of having them:

In [14]:
model = richer.schema_pydantic["employee"]

ok = model(email="e@example.com", status="active", joined="2019-03-01T09:00:00",
           address={"city": "Berlin", "country": "DE"})
print(type(ok.joined).__name__, ok.joined.year, "|", ok.address.city)

for label, bad in (("value outside the set", {"status": "retired"}),
                   ("not a timestamp", {"joined": "one day in March"}),
                   ("nested field missing", {"address": {"city": "Berlin"}})):
    fields = {"email": "e@example.com", "status": "active",
              "joined": "2019-03-01T09:00:00",
              "address": {"city": "Berlin", "country": "DE"}, **bad}
    try:
        model(**fields)
        print(f"!! accepted: {label}")
    except Exception as exc:
        print(f"{label:22} -> {type(exc).__name__}: {exc.errors()[0]['msg']}")

datetime 2019 | Berlin
value outside the set  -> ValidationError: Input should be 'active' or 'gone'
not a timestamp        -> ValidationError: Input should be a valid datetime or date, invalid character in year
nested field missing   -> ValidationError: Field required


### Uniqueness, per type

`unique=True` is the one that has no annotation, because no annotation means it:
it is a statement about the *table*, not about one value. It compiles to a
**partial** unique index — unique among rows of that type only — so a `person` and
a `robot` may hold the same email while two people may not.

This is the constraint Neo4j puts behind an enterprise licence, and it is free
here because Postgres has always had it.

In [15]:
from hopai import NodeType, Property

keyed = graph.in_graph("keyed")
keyed.define_schema(nodes=[
    NodeType("person", properties=[Property("email", "string", unique=True)]),
    NodeType("robot", properties=[Property("email", "string")]),
])
print([statement for statement in keyed.schema_ddl() if "UNIQUE" in statement][0], "\n")
keyed.enforce_schema()

keyed.add_nodes([{"type": "person", "email": "a@example.com"},
                 {"type": "robot", "email": "a@example.com"},   # another type: fine
                 {"type": "person"}])                           # no email: fine, twice over
keyed.add_nodes([{"type": "person"}])

try:
    keyed.add_nodes([{"type": "person", "email": "a@example.com"}])
except ConstraintViolation as exc:
    print(f"{exc.constraint}\n   {exc}")

CREATE UNIQUE INDEX IF NOT EXISTS "uq_schema_78da267198_person_email" ON "nodes" (graph_id, (properties ->> 'email')) WHERE (properties @> CAST('{"type": "person"}' AS JSONB)) 

uq_schema_78da267198_person_email
   node rejected by constraint 'uq_schema_78da267198_person_email' -- Key (graph_id, (properties ->> 'email'::text))=(keyed, a@example.com) already exists.


Rows *missing* the property repeat freely — SQL's `NULL`s are distinct from each
other, and that is `Unique`'s behaviour throughout this library rather than
something the schema layer invents.

On a graph that grew before the rule did, `schema_violations()` answers the
uniqueness question the same way it answers the others — and it reports the **whole
duplicate group**, where the failing `CREATE INDEX` would name one pair and stop:

In [16]:
grew = graph.in_graph("grew-first")
grew.add_nodes([{"type": "contact", "email": "dup@example.com"},
                {"type": "contact", "email": "dup@example.com"},
                {"type": "contact", "email": "dup@example.com"},
                {"type": "contact", "email": "fine@example.com"}])

grew.define_schema(nodes=[NodeType("contact", properties=[
    Property("email", "string", unique=True)])])
print(grew.schema_violations())


3 row(s) violate 1 schema rule(s):
  uq_schema_1aa06b7960_contact_email (nodes): 3 row(s), e.g. id 17, 18, 19


Never inferred, though. `infer_schema()` below reports what the rows *are*, and
every value being distinct today is not a promise that the next write will be —
declaring uniqueness is a decision, not an observation.

## Inferring, when the graph came first

Never declared anything and have a million rows? The schema is sitting in the data
and Postgres can compute it — a few `GROUP BY`s over JSONB.

In [17]:
chaos = graph.in_graph("chaotic")
chaos.add_nodes([
    {"type": "person", "email": "a@example.com", "age": 30},
    {"type": "person", "email": "b@example.com"},                  # no age
    {"type": "person", "email": "c@example.com", "age": "31"},     # age as a string
    {"type": "company", "name": "Acme"},
    {"name": "who knows"},                                         # no type at all
])
chaos.add_edges([{"start": {"email": "a@example.com"}, "end": {"name": "Acme"},
                  "kind": "works_at", "since": 2019}])

inferred, report = chaos.infer_schema()
print(report)

nodes: 4 typed across 2 type(s) {'company': 1, 'person': 3}, 1 untyped (outside the schema)
edges: 1 with a kind across 1 kind(s) {'works_at': 1}, 0 kindless, 0 skipped (endpoint node carries no type)
conflict: nodes/person.age observed as ['number', 'string']


The report is the honest half. A property on *every* row of its type infers
required; missing on some infers optional; an observed null infers nullable; and a
key holding both `42` and `"42"` infers the **type set** `["number", "string"]` plus
a conflict line — never a silently picked winner. Rows with no `type` cannot be
invented into a type, so they are counted and left alone.

### Sampling, when a full scan is too much

`infer_schema()` is a sequential scan per query — fine at start-up, and the
documented cost. On a table too large for that, `sample_percent` reads a
`TABLESAMPLE SYSTEM` slice instead.

It changes what the answer *means*, so the report says so rather than letting an
estimate read as truth: counts become estimates, a rare property or edge triple can
be missed entirely, and `required` weakens to "present on every **sampled** row".

In [18]:
_, sampled = chaos.infer_schema(sample_percent=100)
print(sampled, "\n")

# Out of range is refused before anything connects, naming the range.
try:
    chaos.infer_schema(sample_percent=0)
except ValueError as exc:
    print(exc)

sampled 100% of rows -- counts are estimates
nodes: 4 typed across 2 type(s) {'company': 1, 'person': 3}, 1 untyped (outside the schema)
edges: 1 with a kind across 1 kind(s) {'works_at': 1}, 0 kindless, 0 skipped (endpoint node carries no type)
conflict: nodes/person.age observed as ['number', 'string'] 

sample_percent must satisfy 0 < sample_percent <= 100, got 0


`100` is the honest thing to demo on five rows — `TABLESAMPLE SYSTEM (100)` reads
every page, so the numbers match the exact scan while the report still carries the
flag. At a realistic `5` on a realistic table, they would not.

One deliberate asymmetry: the endpoint-pair join samples the **edges** side only.
Sampling the nodes too would count an edge whose endpoint node happened to fall
outside the sample as endpoint-less, manufacturing `skipped` noise that says
something false about the data rather than merely imprecise.

In [19]:
for node_type in inferred.node_types:
    print(node_type.name, [(p.name, p.json_type, p.required) for p in node_type.properties])
for edge_type in inferred.edge_types:
    print(edge_type.kind, edge_type.source, "->", edge_type.target)

company [('name', ('string',), True)]
person [('age', ('number', 'string'), False), ('email', ('string',), True)]
works_at person -> company


Nothing is registered by that call. Adopting the observation as the contract is a
separate line — which is exactly why `infer_schema()` is a method and not a silent
`.schema` fallback:

```python
inferred, report = graph.infer_schema()
print(report)                             # read this first
graph.define_schema(schema=inferred)      # adopt it -- your call
graph.enforce_schema()                    # chaotic graph, now server-validated
```

Enforcing *that* schema as-is would bless `age` holding both numbers and strings.
Tightening it to `Property("age", "number")` and then running `schema_violations()`
gives you the list of rows to fix first — the loop this API is shaped for.

## Sharing the contract between processes

Everything so far lives on one `Graph` handle in one process. The service next door
gets nothing from it — it would have to declare the same schema again, and the two
declarations would drift the first time one of them changed.

`save_schema()` puts the declared schema in the database, so the database is the
single source of truth for the shape as well as the data:

In [20]:
from hopai import Graph

graph.save_schema()          # upserts one row per graph into hopai_schema

elsewhere = Graph(graph.engine)       # a fresh handle, as another process would build
print(elsewhere.schema, "<- nothing declared on this handle")

loaded = elsewhere.load_schema()
print(loaded == graph.schema, "<- identical, and now adopted")
print([nt.name for nt in elsewhere.schema.node_types])

None <- nothing declared on this handle
True <- identical, and now adopted
['person', 'company']


Three things worth knowing about that table:

- It is **metadata, not graph data** — never on the query path, and created lazily by
  the first `save_schema()`, so a project that never persists never grows it. That is
  why `create_schema()` does not make it.
- Loading **adopts**. Unlike an inferred schema — an observation you choose to accept
  — a saved one was explicitly declared a contract by whoever saved it, so
  `load_schema()` registers it and `enforce_schema()` works straight from the loaded
  copy.
- The stored document is `schema_json` verbatim, readable in `psql`, and it is *data*
  rather than trusted state: loading rebuilds it through the same constructors
  `define_schema()` uses, so a corrupted row raises the real validation error instead
  of half-loading a shape nobody wrote.

---

Next: [07 · Many graphs](07_many_graphs.ipynb) — thousands of isolated graphs in one
pair of tables.
